In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

In [2]:
# 1. Load dataset
df = pd.read_csv("test_data.txt")
df

,Unnamed: 0,ViewYN,PoolPrivateYN,ClosePrice,Latitude,Longitude,LivingArea,CountyOrParish,AttachedGarageYN,ParkingTotal,BathroomsTotalInteger,City,BedroomsTotal,FireplaceYN,Levels,LotSizeArea,NewConstructionYN,HighSchoolDistrict,PostalCode,BuildingAge
0,3,True,False,890000.0,34.264692,-117.221040,3000.0,San Bernardino,True,2.0,3.0,Lake Arrowhead,3.0,True,Two,9600.0,True,Rim of the World,92352,4.0
1,13,True,False,865000.0,33.906058,-117.777782,1442.0,Orange,True,2.0,2.0,Yorba Linda,3.0,True,One,4800.0,False,Placentia-Yorba Linda Unified,92886,40.0
2,18,False,False,720000.0,34.079509,-117.642104,1102.0,San Bernardino,False,2.0,1.0,Ontario,2.0,True,Two,6576.0,False,Chaffey Joint Union High,91764,125.0
3,19,True,False,585000.0,33.918994,-117.477643,1058.0,Riverside,True,6.0,1.0,Riverside,3.0,False,One,7405.0,False,Alvord Unified,92505,70.0
4,22,True,False,559900.0,34.455319,-117.393217,2038.0,San Bernardino,False,2.0,2.0,Victorville,4.0,False,One,7531.0,False,Hesperia Unified,92392,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6733,22758,True,False,905000.0,34.086351,-118.285852,1600.0,Los Angeles,True,2.0,3.0,Los Angeles,2.0,False,ThreeOrMore,2140.0,False,Los Angeles Unified,90029,2.0
6734,22777,True,True,1325000.0,34.240858,-119.026262,3295.0,Ventura,True,5.0,4.0,Camarillo,5.0,True,Two,10747.0,False,NaN,93010,45.0
6735,22791,False,False,578950.0,34.405117,-117.376748,2260.0,San Bernardino,True,2.0,2.0,Hesperia,5.0,False,One,8234.0,True,Hesperia Unified,92344,0.0
6736,22827,True,False,1200000.0,34.300533,-118.704530,2900.0,Ventura,True,2.0,4.0,Simi Valley,4.0,False,Two,8229.0,True,NaN,93063,1.0


In [3]:
# 2. Separate features and target
y = df["ClosePrice"]
X = df.drop(columns=["ClosePrice","HighSchoolDistrict","BuildingAge","Unnamed: 0"])

In [4]:
# 3. Identify numeric and categorical features
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

In [5]:
# 4. Handle missing values (replace with median/most frequent)
# This will be done inside the pipeline using SimpleImputer
from sklearn.impute import SimpleImputer
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),("onehot", OneHotEncoder(handle_unknown="ignore"))])


In [6]:
# 5. Combine preprocessing
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features),("cat", categorical_transformer, categorical_features)])

In [7]:
# 6. Define model
model = RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)

In [8]:
# 7. Build pipeline
pipeline = Pipeline(steps=[("preprocessor", preprocessor),("model", model)])

In [9]:
# 8. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 9. Fit the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Latitude', 'Longitude',
                                                   'LivingArea', 'ParkingTotal',
                                                   'BathroomsTotalInteger',
                                                   'BedroomsTotal',
                                                   'LotSizeArea']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ViewYN', 'PoolPrivateYN',
                                                   'CountyOrParish',
                                                   'AttachedGarageYN', 'City',
                                                   'FireplaceYN', 'Levels',
                                                   'NewConstructionYN',
                                                   'PostalCode'])])),
                ('model',
                 RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [11]:
# 10. Predictions
y_pred = pipeline.predict(X_test)

In [12]:
# Apply natural log to actual and predicted prices
y_test_log = np.log(y_test)
y_pred_log = np.log(y_pred)

In [13]:
# 11. Evaluate
print("Mean Absolute Percentage Error:", mean_absolute_percentage_error(y_test_log, y_pred_log))
print("R² Score:", r2_score(y_test_log, y_pred_log))


Mean Absolute Percentage Error: 0.00947978437435475
R² Score: 0.8881722107212607
